In [1]:
from astrovision.plot.plot_utils import make_mosaic
from astrovision.data import SatelliteImage, SegmentationLabeledSatelliteImage
import s3fs
import os
import geopandas as gpd
from pyproj import CRS
import matplotlib.pyplot as plt
import pandas as pd
import folium
import numpy as np

In [2]:
fs = s3fs.S3FileSystem(client_kwargs={"endpoint_url": "https://" + "minio.lab.sspcloud.fr"})
out  = fs.download(rpath="projet-slums-detection/ilots", lpath="ilots", recursive=True)
out = fs.download(rpath="projet-slums-detection/data-prediction/PLEIADES/MAYOTTE", lpath="pred_mayotte", recursive=True)

pred_2020 = gpd.read_file('pred_mayotte/2020/predictions.gpkg')
pred_2023 = gpd.read_file('pred_mayotte/2023/predictions.gpkg')
ilots = gpd.read_file("ilots/ilots.gpkg")
tgt_crs = CRS.from_epsg(4471) # mayotte 

pred_2020 = pred_2020.to_crs(tgt_crs)
pred_2023 = pred_2023.to_crs(tgt_crs)

pred_2020["area"] = pred_2020.area
pred_2023["area"] = pred_2023.area

ilots = ilots.to_crs(tgt_crs)
ilotsMayotte = ilots[ilots["ident_ilot"].str.startswith('976')]
ilotsMayotte["area"] = ilotsMayotte.geometry.area

In [5]:
intersection_2020 = gpd.overlay(pred_2020, ilotsMayotte, how='intersection')
intersection_2023 = gpd.overlay(pred_2023, ilotsMayotte, how='intersection')

In [13]:
intersection_2020["area_pred_inter_ilot"] = intersection_2020.geometry.area
intersection_2023["area_pred_inter_ilot"] = intersection_2020.geometry.area

aire_inter_2020 = intersection_2020.groupby('ident_ilot')['area_pred_inter_ilot'].sum()
aire_inter_2023 = intersection_2023.groupby('ident_ilot')['area_pred_inter_ilot'].sum()

comp_area = pd.merge(aire_inter_2020,aire_inter_2023,how = 'outer', on = 'ident_ilot', suffixes=('_2020', '_2023'))

comp_area["diff_area"] = comp_area["area_pred_inter_ilot_2023"] - comp_area["area_pred_inter_ilot_2020"]
comp_area["evol_pct_area"] = round((comp_area["diff_area"]/comp_area["area_pred_inter_ilot_2020"])*100,2)

In [16]:
comp_area = comp_area.sort_values(['diff_area'], ascending = False)
comp_area

,area_pred_inter_ilot_2020,area_pred_inter_ilot_2023,diff_area,evol_pct_area
ident_ilot,,,,
976110306,40354.026737,247935.414846,207581.388109,514.40
976070401,19104.200825,111095.817162,91991.616337,481.53
976100112,9528.696758,70793.458717,61264.761959,642.95
976020415,27709.418407,88371.052853,60661.634445,218.92
976170109,14848.251527,69834.738906,54986.487379,370.32
...,...,...,...,...
976110302,72510.707076,50021.500000,-22489.207076,-31.02
976170311,35654.923330,13011.816709,-22643.106621,-63.51
976080241,35742.039815,249.500000,-35492.539815,-99.30


Du coup on veut évaluer graphiquement les raisons des évolutions les plus marquées. 
Pour ce faire on représente sur une meme carte en rouge les créations 2023 ( parts de polygone de pred 2023 non intersectée avec du 2020)
en bleu  les destructions 2023 ( parts de polygone de pred 2020 non intersectée avec du 2023)

In [69]:
sym_diff = pred_2020.symmetric_difference(pred_2023)
sym_diff = pd.DataFrame({'geometry': sym_diff})
sym_diff = gpd.GeoDataFrame(sym_diff, crs='EPSG:4471')


suppression = gpd.sjoin(sym_diff, pred_2020, how='inner', op='intersects')
creation = gpd.sjoin(sym_diff, pred_2023, how='inner', op='intersects')



/tmp/ipykernel_227448/4159814910.py:1: UserWarning: The indices of the two GeoSeries are different.
  sym_diff = pred_2020.symmetric_difference(pred_2023)


/opt/mamba/lib/python3.11/site-packages/IPython/core/interactiveshell.py:3517: FutureWarning: The `op` parameter is deprecated and will be removed in a future release. Please use the `predicate` parameter instead.
  if await self.run_code(code, result, async_=asy):


In [ ]:
creation

In [ ]:

creation = pd.DataFrame({'geometry': creation})
creation = gpd.GeoDataFrame(creation, crs='EPSG:4471')

suppression = pd.DataFrame({'geometry': suppression})
suppression = gpd.GeoDataFrame(suppression, crs='EPSG:4471')


In [61]:
# intersection ilots creation destruction
ilots_inter_creation = gpd.overlay(creation, ilotsMayotte, how='intersection')
ilots_inter_creation["type"] = "creation"

ilots_inter_suppression = gpd.overlay(suppression, ilotsMayotte, how='intersection')
ilots_inter_suppression["type"] = "suppression"

df_diff = pd.concat([
ilots_inter_creation,
ilots_inter_suppression
])

In [62]:
ident_ilot =  "976110306" 
diff_ilot = df_diff.loc[df_diff.ident_ilot  == ident_ilot] 	
contour_ilot  = ilotsMayotte[ilotsMayotte.ident_ilot == ident_ilot]


In [67]:
diff_ilot = diff_ilot.to_crs(4326)
contour_ilot = contour_ilot.to_crs(4326)

center = list(contour_ilot.iloc[0]['geometry'].centroid.coords)[0]
center =  (center[1],center[0])

carte = folium.Map(location=center, zoom_start=10)

folium.GeoJson(contour_ilot,style_function=lambda x: {"color": "yellow"}).add_to(carte)

for index, row in sym_diff.iterrows():
    folium.GeoJson(row["geometry"],  style_function=lambda x: {"color": "orange"}).add_to(carte)

# Ajouter les polygones de type "creation" en rouge
for index, row in diff_ilot[diff_ilot["type"] == "creation"].iterrows():
    folium.GeoJson(row["geometry"],  style_function=lambda x: {"color": "red"}, tooltip=row["ident_ilot"]).add_to(carte)

# Ajouter les polygones de type "suppression" en bleu
for index, row in diff_ilot[diff_ilot["type"] == "suppression"].iterrows():
    folium.GeoJson(row["geometry"], style_function=lambda x: {"color": "blue"}, tooltip=row["ident_ilot"]).add_to(carte)


/tmp/ipykernel_227448/2707524013.py:4: UserWarning: The indices of the two GeoSeries are different.
  sym_diff = pred_2020.symmetric_difference(pred_2023)


KeyboardInterrupt: 

In [64]:
carte